In [1]:
### Set parameters
# model_type = 'multil'
# model_type = 'sonoisa'
# model_type = 'tohoku'
# model_type = 'izumil'
model_type = 'multil_jppost-cp-e1-b512'

train_data_type = '100'
# train_data_type = '090'

epochs_int = 5
batch_size_int = 1

print(f'{model_type = }')
print(f'{train_data_type = }')
print(f'{epochs_int = }')
print(f'{batch_size_int = }')

model_type = 'multil_jppost-cp-e1-b512'
train_data_type = '100'
epochs_int = 5
batch_size_int = 1


In [2]:
import os

### Set the model name
model_base = ""
if model_type == 'multil':
    model_name = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
elif model_type == 'sonoisa':
    model_name = 'sonoisa/sentence-bert-base-ja-mean-tokens-v2'
elif model_type == 'tohoku':
    model_name = 'cl-tohoku/bert-base-japanese-whole-word-masking'
elif model_type == 'izumil':
    model_name = 'izumi-lab/bert-small-japanese'
else:
    model_base = './tuned_models/'
    model_name = model_base + model_type

model_name = model_base + "multil_jppost-cp-e1to3-b512/checkpoints/4842"
print(f'{model_name = }')

### Set the directory path for output and create the directory
# model_save_path = './tuned_models/' + model_type + '_' + datetime.now().strftime("%Y%m%d_%H%M%S")
model_save_path = './tuned_models/' + model_type + '_jsts-' + train_data_type + '-cp-e1to' + str(epochs_int) + '-b' + str(batch_size_int)
checkpoint_save_path = model_save_path + '/checkpoints'

print(f'{model_save_path = }')
print(f'{checkpoint_save_path = }')

os.mkdir(model_save_path)
os.mkdir(checkpoint_save_path)

model_name = './tuned_models/multil_jppost-cp-e1to3-b512/checkpoints/4842'
model_save_path = './tuned_models/multil_jppost-cp-e1-b512_jsts-100-cp-e1to5-b1'
checkpoint_save_path = './tuned_models/multil_jppost-cp-e1-b512_jsts-100-cp-e1to5-b1/checkpoints'


In [3]:
import json
import pandas as pd
from sentence_transformers import InputExample


def json_dataset_to_df(filepath):
    data_dicts_list = list()
    
    with open(filepath) as fp:
        for line_str in fp:
            line_dict = json.loads(line_str)
            data_dicts_list.append(line_dict)
    
    data_df = pd.DataFrame(data_dicts_list)
    
    return data_df


def create_data_training_list(data_train_df):
    data_train_inputexamples_list = list()
    
    for index, row in data_train_df.iterrows():
        sentence1_str = row['sentence1']
        sentence2_str = row['sentence2']
        label_float = float(row['label']) / 5
        data_train_inputexamples_list.append(InputExample(texts=[sentence1_str, sentence2_str], label=label_float))
    
    return data_train_inputexamples_list


def create_data_validation_dict(data_df):
    sentences_1_list = data_df['sentence1'].tolist()
    sentences_2_list = data_df['sentence2'].tolist()
    labels_sr = data_df['label'].astype('float') / 5
    labels_list = labels_sr.tolist()
        
    data_valid_lists_dict = {
        'sentence1': sentences_1_list,
        'sentence2': sentences_2_list,
        'label': labels_list
    }
    return data_valid_lists_dict


In [4]:
# Create datasets for training, validation and test.
import math
from sklearn.model_selection import train_test_split

FILEPATH_INPUT_TRAIN = '../datasets/jsts-v1.1/train-v1.1.json'
FILEPATH_INPUT_TEST = '../datasets/jsts-v1.1/valid-v1.1.json'


# Read the training dataset file and convert them to the dataframes.
data_train_read_df = json_dataset_to_df(FILEPATH_INPUT_TRAIN)
print(data_train_read_df.head(3))

# Divide data into training one and validation.
data_train_df = pd.DataFrame()
if train_data_type == '100':
    data_train_df = data_train_read_df.copy()
if train_data_type == '090':
    data_train_df, data_valid_df = train_test_split(data_train_read_df, test_size=0.1, shuffle=False)

# Format the data for training and validation.
data_train_inputexamples_list = create_data_training_list(data_train_df)
length_data_train_int = len(data_train_inputexamples_list)
print(length_data_train_int)
print(data_train_inputexamples_list[:3])


# Read the training dataset file and convert them to the dataframes.
data_test_df = json_dataset_to_df(FILEPATH_INPUT_TEST)
print(data_test_df.head(3))

# Create datasets for test.
data_valid_lists_dict = create_data_validation_dict(data_test_df)

print(len(data_valid_lists_dict['sentence1']))
print(data_valid_lists_dict['sentence1'][:3])
print(len(data_valid_lists_dict['sentence2']))
print(data_valid_lists_dict['sentence2'][:3])
print(len(data_valid_lists_dict['label']))
print(data_valid_lists_dict['label'][:3])


steps_per_epoch_float = length_data_train_int / batch_size_int

### Calculate warmup steps
warmup_steps_int = steps_per_epoch_float // 100
print(f'{warmup_steps_int = }')

### Calculate checkpoint steps
checkpoint_steps_int = math.ceil(steps_per_epoch_float)
print(f'{checkpoint_steps_int = }')

  sentence_pair_id             yjcaptions_id               sentence1   
0                0  10005_480798-10996-92616  川べりでサーフボードを持った人たちがいます。  \
1                1      100124-104404-104405  二人の男性がジャンボジェット機を見ています。   
2                2      100142-104431-104432      男性が子供を抱き上げて立っています。   

               sentence2  label  
0  トイレの壁に黒いタオルがかけられています。    0.0  
1   2人の男性が、白い飛行機を眺めています。    3.8  
2   坊主頭の男性が子供を抱いて立っています。    4.0  
12451
[<sentence_transformers.readers.InputExample.InputExample object at 0x7f7d2d4c73a0>, <sentence_transformers.readers.InputExample.InputExample object at 0x7f7d2d4c78e0>, <sentence_transformers.readers.InputExample.InputExample object at 0x7f7d2d4c7d30>]
  sentence_pair_id               yjcaptions_id                    sentence1   
0                0  100312_421853-104611-31624  レンガの建物の前を、乳母車を押した女性が歩いています。  \
1                1        100371-104675-104678             山の上に顔の白い牛が2頭います。   
2                2        100668-104946-104949         バナナを持った人が道路を通行しています。   


In [ ]:
# Get a pre-trained model
from sentence_transformers import SentenceTransformer, losses, models

transformer = models.Transformer(model_name)

pooling = models.Pooling(transformer.get_word_embedding_dimension(), pooling_mode='mean')
model = SentenceTransformer(modules=[transformer, pooling], device='cuda')

# Define your train dataset, the dataloader and the train loss
train_loss = losses.CosineSimilarityLoss(model)

In [ ]:
from sentence_transformers import evaluation
from torch.utils.data import DataLoader


dataloader_train = DataLoader(
    dataset = data_train_inputexamples_list,
    batch_size = batch_size_int,
    shuffle = True,  # False
    sampler = None,
    batch_sampler = None,
    num_workers = 0,
    collate_fn = None,
    pin_memory = False,
    drop_last = False,
    timeout = 0,
    worker_init_fn = None,
    prefetch_factor = 2,
    persistent_workers = False
)
evaluator = evaluation.EmbeddingSimilarityEvaluator(
    sentences1 = data_valid_lists_dict['sentence1'],
    sentences2 = data_valid_lists_dict['sentence2'],
    scores = data_valid_lists_dict['label'],
    batch_size = 1,  # 16
    main_similarity = evaluation.SimilarityFunction.COSINE,
    name = '',
    show_progress_bar = True,
    write_csv = True
)
model.fit(
    train_objectives = [(dataloader_train, train_loss)],
          evaluator = evaluator,
          epochs = epochs_int,
          steps_per_epoch = None,
          scheduler = 'WarmupLinear',
          warmup_steps = warmup_steps_int,  # 10000
          optimizer_params = {'lr': 2e-05},  # <class 'torch.optim.adamw.AdamW'>
          weight_decay = 0.01,
          evaluation_steps = 0,
          output_path = model_save_path,
          save_best_model = True,
          max_grad_norm = 1,
          use_amp = False,
          callback = None,
          show_progress_bar = True,
          checkpoint_path = checkpoint_save_path,  # None
          checkpoint_save_steps = checkpoint_steps_int,  # 500
          checkpoint_save_total_limit = epochs_int  # 0
)


In [7]:
### tohoku_jppost-e1-b512_jsts-e1-b1
# score_pearson_float = 0.8669176916978663
# score_spearman_float = 0.8201534801572957

### tohoku_jppost-e1-b512_jsts-e2-b1
# score_pearson_float = 0.8673423410964516
# score_spearman_float = 0.8201792413229232

### tohoku_jppost-e1-b512_jsts-e3-b1
# score_pearson_float = 0.8658976143340302
# score_spearman_float = 0.8207110349635123

### tohoku_jppost-e1-b512_jsts-e4-b1
# score_pearson_float = 0.8765113034171279
# score_spearman_float = 0.8314736418265504

### tohoku_jppost-e1-b512_jsts-e5-b1
# score_pearson_float = 0.8729439398135683
# score_spearman_float = 0.831210761326818


### izumil_jppost-e1-b512_jsts-e3-b1
# score_pearson_float = 0.850063214198677
# score_spearman_float = 0.7991509353134065

### cl_tohoku_20230420_013801_jppost_20230420_021605
# score_pearson_float = 0.8719925090234125
# score_spearman_float = 0.8272646327420434